In [6]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml import command, load_component
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import Input, Output
from azureml.core import Workspace, Dataset, Datastore

subscription_id = '813ac1cb-7b8e-45ab-9b3c-8510079c761a'
resource_group = 'MIBLab_Official'
workspace_name = 'miblab-official-ml'

workspace = Workspace(subscription_id, resource_group, workspace_name)
datastore = Datastore.get(workspace, "eegfoundationmodeldata")

# Get a handle to the workspace
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)

cluster_name = "miblab-Standard-NC16as-T4-v3-gpu"
download_component = load_component(source="./DownloadHBN.yml")

In [7]:
openneuro_accession_number = Input(
	type="string",
)

max_concurrent_downloads = Input(
	type="integer",
	default=5,
)


In [8]:
output_dir = Output(
	type="uri_folder",
	path='azureml://subscriptions/813ac1cb-7b8e-45ab-9b3c-8510079c761a/resourcegroups/MIBLab_Official/workspaces/miblab-official-ml/datastores/eegfoundationmodeldata/paths/eegdata/ds004186-download',
	mode='rw_mount'
)

In [9]:
# Define a pipeline with component
@pipeline(default_compute=cluster_name)
def download(
	openneuro_accession_number,
	max_concurrent_downloads,
    ):
    
	download_comp = download_component(
		openneuro_accession_number=openneuro_accession_number,
		max_concurrent_downloads=max_concurrent_downloads,
	)

	return {
        "output_dir": download_comp.outputs.output_dir
    }

In [10]:
download_job = download(openneuro_accession_number="ds004186",
						max_concurrent_downloads=8
			)

download_job.outputs.output_dir = output_dir
# submit job to workspace
download_job = ml_client.jobs.create_or_update(
	download_job,
    experiment_name="EEGFoundationModel",
)

download_job

Uploading DataAcquisition (0.01 MBs): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14602/14602 [00:00<00:00, 475657.83it/s]


Experiment,Name,Type,Status,Details Page
EEGFoundationModel,lucid_band_tc2z92dptx,pipeline,NotStarted,Link to Azure Machine Learning studio


In [1]:
import openneuro

In [3]:
openneuro.download(dataset="ds005505", target_dir="/home/azureuser/mycontainer/HBN/ds005505/", max_concurrent_downloads=1)


👋 Hello! This is openneuro-py 2024.2.0. Great to see you! 🤗

   👉 Please report problems 🤯 and bugs 🪲 at
      https://github.com/hoechenberger/openneuro-py/issues

🌍 Preparing to download ds005505 …


📁 Traversing directories for ds005505 : 0 entities [00:00, ? entities/s]

📥 Retrieving up to 5406 files (1 concurrent downloads). 
✅ Finished downloading ds005505.
 
🧠 Please enjoy your brains.
 
